# 1

In [2]:
import os
import logging
import asyncio
from datetime import datetime, timezone, timedelta

from dotenv import load_dotenv
from rich.logging import RichHandler

import yfinance as yf
from openai import AsyncAzureOpenAI
from agents import (
    Agent,
    OpenAIChatCompletionsModel,
    Runner,
    function_tool,
    set_tracing_disabled,
)

# ----- Logging bonito en consola -----
logging.basicConfig(level=logging.INFO, format="%(message)s", handlers=[RichHandler()])

# Cargar variables del .env
load_dotenv()

# Desactivar tracing si no lo usas
set_tracing_disabled(True)

# ----- Azure OpenAI Configuration -----
endpoint = os.getenv("ENDPOINT_URL")
deployment = os.getenv("DEPLOYMENT_NAME")
subscription_key = os.getenv("AZURE_OPENAI_API_KEY")
api_version = os.getenv("OPENAI_API_VERSION", "2025-01-01-preview")

if not all([endpoint, deployment, subscription_key]):
    raise RuntimeError(
        "Faltan variables de entorno: ENDPOINT_URL, DEPLOYMENT_NAME o AZURE_OPENAI_API_KEY."
    )

client = AsyncAzureOpenAI(
    api_version=api_version,
    azure_endpoint=endpoint,
    api_key=subscription_key,
)

MODEL_NAME = deployment


# 2

In [ ]:
def get_current_time() -> str:
    return datetime.now(timezone.utc).strftime("%Y-%m-%d")

@function_tool
def get_current_date() -> str:
    return get_current_time()

In [3]:
import os
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from google.auth.transport.requests import Request

# Los permisos que el agente necesita (leer y escribir eventos)
SCOPES = ['https://www.googleapis.com/auth/calendar.events']

def authenticate_google_calendar():
    creds = None
    
    # Verifica si ya existe el token.json de una sesión anterior
    if os.path.exists('token.json'):
        creds = Credentials.from_authorized_user_file('token.json', SCOPES)
    
    # Si no hay credenciales válidas o expiraron, te pedirá iniciar sesión
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            # AQUÍ es donde tu código lee el archivo credentials.json que acabas de descargar
            flow = InstalledAppFlow.from_client_secrets_file('credentials.json', SCOPES)
            
            # Esto abrirá una pestaña en tu navegador web
            creds = flow.run_local_server(port=0)
        
        # Guarda el token generado para que el agente lo use automáticamente después
        with open('token.json', 'w') as token:
            token.write(creds.to_json())
            
    print("¡Autenticación exitosa! El archivo token.json se ha creado correctamente.")
    return creds

# Ejecutamos la función de autenticación
authenticate_google_calendar()

Please visit this URL to authorize this application: https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=1081828649459-1o19ror126r669iudbmdl6u62lcagd8u.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A64770%2F&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcalendar.events&state=rKBmrAaSoyNnAbjw6vxLLdOcXNqa23&access_type=offline


[02/25/26 17:33:31] INFO     "GET                                                                       flow.py:476
                             /?state=rKBmrAaSoyNnAbjw6vxLLdOcXNqa23&iss=https://accounts.google.com&cod            
                             e=4/0AfrIepC2XTPSWn7pctHHlFef2FVdQGzqA8kr3kJM4EEiN5Ll_f36-Lk1PxpyrtLJRgQUc            
                             A&scope=https://www.googleapis.com/auth/calendar.events HTTP/1.1" 200 65              

¡Autenticación exitosa! El archivo token.json se ha creado correctamente.
